[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/pinecone-io/examples/blob/master/docs/quick-tour/hello-pinecone.ipynb) [![Open nbviewer](https://raw.githubusercontent.com/pinecone-io/examples/master/assets/nbviewer-shield.svg)](https://nbviewer.org/github/pinecone-io/examples/blob/master/docs/quick-tour/hello-pinecone.ipynb)

# Hello, Pinecone!

This notebook will walk through the steps to get a simple Pinecone index up and running.


## Prerequisites

First we need to install a few dependencies

In [1]:
!pip install -qU pandas==2.2.3 pinecone==8.0.0


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## Getting started

We begin by instantiating an instance of the Pinecone client. To do this we need a [free API key](https://app.pinecone.io).

In [2]:
import os
from pinecone import Pinecone

# Get your API key at app.pinecone.io
api_key = os.environ.get("PINECONE_API_KEY") or "pcsk_43oqMm_EAbh5zhm7Lhr3hYXMzQtesXYjMw31KgkuAzCMh3CkiQk1BtVxrqtcqRzz7P7cUk"

# Instantiate the Pinecone client
pc = Pinecone(api_key=api_key)

## Creating an index

With Pinecone you can create a vector index where you can store and search through your vector embeddings.

In [3]:
# Giving our index a name
index_name = "hello-pinecone"

In [4]:
# Delete the index if an index of the same name already exists
if pc.has_index(name=index_name):
    pc.delete_index(name=index_name)

### Creating a Pinecone Index

When creating the index we need to define several configuration properties. 

- `name` can be anything we like. The name is used as an identifier for the index when performing other operations such as `describe_index`, `delete_index`, and so on. 
- `metric` specifies the similarity metric that will be used later when you make queries to the index.
- `dimension` should correspond to the dimension of the dense vectors produced by your embedding model. In this quick start, we are using made-up data so a small value is simplest.
- `spec` holds a specification which tells Pinecone how you would like to deploy our index. You can find a list of all [available providers and regions here](https://docs.pinecone.io/guides/index-data/create-an-index#cloud-regions).

There are more configurations available, but this minimal set will get us started.

In [5]:
from pinecone import ServerlessSpec, CloudProvider, AwsRegion, Metric

pc.create_index(
    name=index_name,
    metric=Metric.COSINE,
    dimension=3,
    spec=ServerlessSpec(cloud=CloudProvider.AWS, region=AwsRegion.US_EAST_1),
)

{
    "name": "hello-pinecone",
    "metric": "cosine",
    "host": "hello-pinecone-mpjmug0.svc.aped-4627-b74a.pinecone.io",
    "spec": {
        "serverless": {
            "region": "us-east-1",
            "cloud": "aws",
            "read_capacity": {
                "mode": "OnDemand",
                "status": {
                    "state": "Ready",
                    "current_shards": null,
                    "current_replicas": null
                }
            }
        }
    },
    "status": {
        "ready": true,
        "state": "Ready"
    },
    "vector_type": "dense",
    "dimension": 3,
    "deletion_protection": "disabled",
    "tags": null,
    "_response_info": {
        "raw_headers": {
            "content-type": "application/json",
            "vary": "origin, access-control-request-method, access-control-request-headers",
            "access-control-allow-origin": "*",
            "access-control-expose-headers": "*",
            "x-pinecone-api-version": "

We can look up the configuration for the index anytime we like by using `describe_index`

In [6]:
description = pc.describe_index(name=index_name)
description

{
    "name": "hello-pinecone",
    "metric": "cosine",
    "host": "hello-pinecone-mpjmug0.svc.aped-4627-b74a.pinecone.io",
    "spec": {
        "serverless": {
            "region": "us-east-1",
            "cloud": "aws",
            "read_capacity": {
                "mode": "OnDemand",
                "status": {
                    "state": "Ready",
                    "current_shards": null,
                    "current_replicas": null
                }
            }
        }
    },
    "status": {
        "ready": true,
        "state": "Ready"
    },
    "vector_type": "dense",
    "dimension": 3,
    "deletion_protection": "disabled",
    "tags": null,
    "_response_info": {
        "raw_headers": {
            "content-type": "application/json",
            "vary": "origin, access-control-request-method, access-control-request-headers",
            "access-control-allow-origin": "*",
            "access-control-expose-headers": "*",
            "x-pinecone-api-version": "

## Upserting data into the index

We can see the index ready. Now we will create some simple vectors that will serve as our examples.

In [7]:
# Instantiate an Index client
index = pc.Index(host=description.host)

In [8]:
import random
import pandas as pd

# Set seed for reproducible results
random.seed(42)


def create_simulated_data_in_df(num_vectors):
    df = pd.DataFrame(
        data={
            "id": [f"id-{i}" for i in range(num_vectors)],
            "vector": [
                [random.random() for _ in range(description.dimension)]
                for _ in range(num_vectors)
            ],
        }
    )
    return df


df = create_simulated_data_in_df(10)

df.head()

,id,vector
0,id-0,"[0.6394267984578837, 0.025010755222666936, 0.2..."
1,id-1,"[0.22321073814882275, 0.7364712141640124, 0.67..."
2,id-2,"[0.8921795677048454, 0.08693883262941615, 0.42..."
3,id-3,"[0.029797219438070344, 0.21863797480360336, 0...."
4,id-4,"[0.026535969683863625, 0.1988376506866485, 0.6..."


We perform `upsert` operations in our index. This call will insert a new vector in the index or update the vector if the id was already present.

In [9]:
index.upsert(vectors=zip(df.id, df.vector))  # insert vectors

UpsertResponse(upserted_count=10, _response_info={'raw_headers': {'date': 'Thu, 05 Feb 2026 19:40:29 GMT', 'content-type': 'application/json', 'content-length': '20', 'connection': 'keep-alive', 'x-pinecone-request-lsn': '1', 'x-pinecone-request-logical-size': '160', 'x-pinecone-request-latency-ms': '248', 'x-envoy-upstream-service-time': '248', 'x-pinecone-response-duration-ms': '252', 'grpc-status': '0', 'server': 'envoy'}})

In [10]:
import time

# Wait for vectors to be indexed and available for querying
while index.describe_index_stats().total_vector_count == 0:
    time.sleep(1)

print("Vectors are ready for querying!")

Vectors are ready for querying!


In [11]:
# View index stats
index.describe_index_stats()

{'_response_info': {'raw_headers': {'connection': 'keep-alive',
                                    'content-length': '181',
                                    'content-type': 'application/json',
                                    'date': 'Thu, 05 Feb 2026 19:40:29 GMT',
                                    'grpc-status': '0',
                                    'server': 'envoy',
                                    'x-envoy-upstream-service-time': '45',
                                    'x-pinecone-request-latency-ms': '44',
                                    'x-pinecone-response-duration-ms': '47'}},
 'dimension': 3,
 'index_fullness': 0.0,
 'memoryFullness': 0.0,
 'metric': 'cosine',
 'namespaces': {'__default__': {'vector_count': 10}},
 'storageFullness': 0.0,
 'total_vector_count': 10,
 'vector_type': 'dense'}

## Running a query

Next we can run a query.

In a more realistic scenario, the `vector` values passing into `query` would be an embedding vector of something meaningful. But for this simple walkthrough we will use made up values. The query will succeed as long as the dimension matches the dimension of our index.

`top_k` specifies the number of results we would like returned. The method will return up to `top_k` results, but may be less if there are fewer than `top_k` vectors in your index or if all indexes have been filtered out using metadata filters.

In [12]:
# In a more realistic scenario, this would be an embedding vector
# that encodes something meaningful. For this simple demo, we will
# make up a vector that matches the dimension of our index.
query_embedding = [2.0, 2.0, 2.0]

index.query(vector=query_embedding, top_k=5, include_values=True)

QueryResponse(matches=[{'id': 'id-5',
 'score': 0.939291835,
 'values': [0.544941485, 0.220440626, 0.589265704]}, {'id': 'id-1',
 'score': 0.921592534,
 'values': [0.223210737, 0.736471236, 0.67669946]}, {'id': 'id-7',
 'score': 0.870520413,
 'values': [0.698139369, 0.340250522, 0.155479506]}, {'id': 'id-9',
 'score': 0.854860187,
 'values': [0.0967163742, 0.847494364, 0.603726]}, {'id': 'id-6',
 'score': 0.820000112,
 'values': [0.80943048, 0.00649875961, 0.805819273]}], namespace='', usage={'read_units': 1}, _response_info={'raw_headers': {'date': 'Thu, 05 Feb 2026 19:40:29 GMT', 'content-type': 'application/json', 'content-length': '468', 'connection': 'keep-alive', 'x-pinecone-max-indexed-lsn': '1', 'x-pinecone-request-latency-ms': '2', 'x-envoy-upstream-service-time': '3', 'x-pinecone-response-duration-ms': '4', 'grpc-status': '0', 'server': 'envoy'}})

## Delete the Index
Delete the index once you are sure that you do not want to use it anymore. **Deletion is permanent**. Once the index is deleted, you cannot use it again.

In [13]:
pc.delete_index(name=index_name)

## Next steps

Now that you've seen the basics, explore more:

- [Semantic search](https://docs.pinecone.io/guides/search/semantic-search) - Search with real embeddings
- [Metadata filtering](https://docs.pinecone.io/guides/search/filter-by-metadata) - Filter results by metadata
- [RAG Getting Started](https://github.com/pinecone-io/examples/blob/master/docs/rag-getting-started.ipynb) - Build a retrieval-augmented generation app